In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.1 MB/s eta 0:00:00


In [ ]:
yaml_content = """
path: /content/drive/MyDrive/RDD2022
train: train/images
val: val/images
test: test/images

nc: 4
names: ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
"""

with open('rdd2022.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("rdd2022.yaml file created successfully.")

rdd2022.yaml file created successfully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Recreate yaml pointing to Drive
yaml_content = """
path: /content/drive/MyDrive/RDD2022
train: train/images
val: val/images

nc: 4
names: ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
"""
with open('rdd2022.yaml', 'w') as f:
    f.write(yaml_content)

print("YAML created, starting validation...")

!yolo val model=/content/drive/MyDrive/RDD2022_runs/baseline_10k-2/weights/best.pt \
          data=rdd2022.yaml \
          imgsz=640 \
          split=val \
          plots=True

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
YAML created, starting validation...
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 6.8±12.3 ms, read: 0.2±0.3 MB/s, size: 195.4 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/1hO8gveIrS95PGSfY57_-zI1mG86yAPqG/RDD2022/val/labels.cache... 5758 images, 1837 backgrounds, 544 corrupt: 100% ━━━━━━━━━━━━ 5758/5758 779.1Mit/s 0.0s
val: /content/drive/.shortcut-targets-by-id/1hO8gveIrS95PGSfY57_-zI1mG86yA

In [ ]:
import shutil
shutil.copytree('/content/runs/detect/val',
                '/content/drive/MyDrive/RDD2022_runs/val_results')
print("Copied to Drive!")

Copied to Drive!


In [ ]:
#Testing if colab can read the images
!ls /content/drive/MyDrive/RDD2022/train/images | head -n 5

China_Drone_000001.jpg
China_Drone_000002.jpg
China_Drone_000003.jpg
China_Drone_000005.jpg
China_Drone_000009.jpg


In [ ]:
import os, shutil, random

# Source paths s
train_src_img = '/content/drive/MyDrive/RDD2022/train/images'
val_src_img = '/content/drive/MyDrive/RDD2022/val/images'

# Destination paths on local Colab storage
dst_dir = '/content/RDD2022_small'
os.makedirs(f'{dst_dir}/train/images', exist_ok=True)
os.makedirs(f'{dst_dir}/train/labels', exist_ok=True)
os.makedirs(f'{dst_dir}/val/images', exist_ok=True)
os.makedirs(f'{dst_dir}/val/labels', exist_ok=True)

#  random sample of 10,000 Train Images
train_images = random.sample(os.listdir(train_src_img), min(10000, len(os.listdir(train_src_img))))
for img in train_images:
    shutil.copy(f'{train_src_img}/{img}', f'{dst_dir}/train/images/{img}')
    lbl = img.replace('.jpg', '.txt').replace('.png', '.txt').replace('.JPEG', '.txt')
    lbl_src = f'{train_src_img.replace("images","labels")}/{lbl}'
    if os.path.exists(lbl_src):
        shutil.copy(lbl_src, f'{dst_dir}/train/labels/{lbl}')

# random sample of 1000 Validation Images
val_images = random.sample(os.listdir(val_src_img), min(1000, len(os.listdir(val_src_img))))
for img in val_images:
    shutil.copy(f'{val_src_img}/{img}', f'{dst_dir}/val/images/{img}')
    lbl = img.replace('.jpg', '.txt').replace('.png', '.txt').replace('.JPEG', '.txt')
    lbl_src = f'{val_src_img.replace("images","labels")}/{lbl}'
    if os.path.exists(lbl_src):
        shutil.copy(lbl_src, f'{dst_dir}/val/labels/{lbl}')

print("Train (10000) and Val (1000) subsets successfully copied locally!")

Train (10000) and Val (1000) subsets successfully copied locally!


In [ ]:
yaml_content = """
path: /content/RDD2022_small
train: train/images
val: val/images

nc: 4
names: ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
"""
with open('rdd2022.yaml', 'w') as f:
    f.write(yaml_content)

In [ ]:
!yolo train model=yolov8s.pt data=rdd2022.yaml epochs=50 imgsz=640 batch=16 lr0=0.01 lrf=0.01 cos_lr=True patience=20 project=/content/drive/MyDrive/RDD2022_runs name=baseline_10k save_period=10

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=rdd2022.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_10k-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=20, perspecti

In [ ]:
# --- EVALUATION METRICS --

import pandas as pd
from IPython.display import Image, display
import os

run_dir = '/content/drive/MyDrive/RDD2022_runs/baseline_10k-2'

if os.path.exists(run_dir):
    print("CONFUSION MATRIX ")
    if os.path.exists(f'{run_dir}/confusion_matrix.png'):
        display(Image(filename=f'{run_dir}/confusion_matrix.png', width=600))

    print("\nF1 SCORE CURVE ")
    if os.path.exists(f'{run_dir}/F1_curve.png'):
        display(Image(filename=f'{run_dir}/F1_curve.png', width=600))

    print("\nPRECISION-RECALL (PR) CURVE ")
    if os.path.exists(f'{run_dir}/PR_curve.png'):
        display(Image(filename=f'{run_dir}/PR_curve.png', width=600))

    print("\nRAW METRICS (Validation) ")
    results_csv = f'{run_dir}/results.csv'
    if os.path.exists(results_csv):
        # Read results and clean up column names
        df = pd.read_csv(results_csv)
        df.columns = df.columns.str.strip()

        # Display the final epoch's metrics
        final_epoch = df.iloc[-1]
        print(f"Final Epoch: {final_epoch['epoch']}")
        print(f"mAP@50:      {final_epoch['metrics/mAP50(B)']:.4f}")
        print(f"mAP@50-95:   {final_epoch['metrics/mAP50-95(B)']:.4f}")
        print(f"Precision:   {final_epoch['metrics/precision(B)']:.4f}")
        print(f"Recall:      {final_epoch['metrics/recall(B)']:.4f}")
else:
    print("Training run directory not found. Please train the model first.")
